In [1]:
from pathlib import Path
import json
import pandas as pd

project_folder = Path(r"D:\electricity-tariff-optimisation")
processed_folder = project_folder / "data" / "processed"

config = json.loads(
    (
        processed_folder / "locked_optional_switch_tariff_2013.json"
    ).read_text(encoding="utf-8")
)

assignments = pd.read_csv(
    processed_folder / "locked_customer_tariff_assignments_2013.csv",
    index_col="household_id",
)

validation_energy = pd.read_csv(
    processed_folder / "customer_validation_band_kwh_2013.csv",
    index_col="household_id",
)

saved_results = pd.read_csv(
    processed_folder / "customer_tariff_results_2013.csv",
    index_col="household_id",
)

for table in [assignments, validation_energy, saved_results]:
    assert table.index.is_unique
    assert len(table) == 248

assert set(assignments.index) == set(validation_energy.index)
assert set(assignments.index) == set(saved_results.index)

assert assignments["assigned_tariff"].isin(
    ["Flat", "Time-of-use"]
).all()

print("Validation inputs loaded.")
print(f"Households: {len(assignments)}")
print("Household keys: PASS")
print("Saved prices and assignments loaded without modification.")
print("Billing calculations have not yet been checked.")

Validation inputs loaded.
Households: 248
Household keys: PASS
Saved prices and assignments loaded without modification.
Billing calculations have not yet been checked.


In [2]:
from decimal import Decimal

bands = ["Peak", "Shoulder", "Off-peak"]

flat_rate = Decimal(str(config["baseline_rate_gbp_per_kwh"]))

tou_rates = {
    band: Decimal(str(config["rates_gbp_per_kwh"][band]))
    for band in bands
}

cost_rates = {
    band: Decimal(str(config["variable_cost_rates_gbp_per_kwh"][band]))
    for band in bands
}

cap_multiplier = (
    Decimal("1")
    + Decimal(str(config["validation_bill_increase_limit_pct"]))
    / Decimal("100")
)

recalculated_rows = []

for household_id in assignments.index:
    energy = {
        band: Decimal(
            str(validation_energy.loc[household_id, band])
        )
        for band in bands
    }

    baseline = sum(energy.values()) * flat_rate

    tou_charge = sum(
        energy[band] * tou_rates[band]
        for band in bands
    )

    assigned_tariff = assignments.loc[
        household_id, "assigned_tariff"
    ]

    assigned_charge = (
        tou_charge
        if assigned_tariff == "Time-of-use"
        else baseline
    )

    variable_cost = sum(
        energy[band] * cost_rates[band]
        for band in bands
    )

    rebate = max(
        Decimal("0"),
        assigned_charge - baseline * cap_multiplier,
    )

    recalculated_rows.append({
        "household_id": household_id,
        "baseline_charge_gbp": float(baseline),
        "tou_charge_gbp": float(tou_charge),
        "assigned_charge_gbp": float(assigned_charge),
        "saving_gbp": float(baseline - assigned_charge),
        "variable_cost_gbp": float(variable_cost),
        "rebate_gbp": float(rebate),
        "protected_charge_gbp": float(assigned_charge - rebate),
    })

recalculated = pd.DataFrame(
    recalculated_rows
).set_index("household_id")

comparison_rows = []

for column in recalculated.columns:
    saved_column = pd.to_numeric(
        saved_results[column], errors="raise"
    ).reindex(recalculated.index)

    assert saved_column.notna().all()

    differences = (
        recalculated[column] - saved_column
    ).abs()

    comparison_rows.append({
        "field": column,
        "max_absolute_difference_gbp": differences.max(),
        "households_outside_tolerance": int(
            (differences > 0.000001).sum()
        ),
    })

billing_checks = pd.DataFrame(comparison_rows)

assignment_match = (
    assignments["assigned_tariff"]
    == saved_results["assigned_tariff"].reindex(assignments.index)
).all()

display(billing_checks)

assert assignment_match, "Saved tariff assignments do not match."
assert (
    billing_checks["households_outside_tolerance"] == 0
).all(), "Billing differences require investigation."

print("Saved tariff assignments: PASS")
print("Independent decimal billing reconciliation: PASS")
print("Scope: billing from saved band totals, not timestamp classification.")

,field,max_absolute_difference_gbp,households_outside_tolerance
0,baseline_charge_gbp,2.273737e-13,0
1,tou_charge_gbp,4.547474e-13,0
2,assigned_charge_gbp,4.547474e-13,0
3,saving_gbp,7.673862e-13,0
4,variable_cost_gbp,2.273737e-13,0
5,rebate_gbp,3.996803e-14,0
6,protected_charge_gbp,4.547474e-13,0


Saved tariff assignments: PASS
Independent decimal billing reconciliation: PASS
Scope: billing from saved band totals, not timestamp classification.


In [3]:
time_cases = pd.DataFrame([
    ("Winter off-peak end", "2013-01-07 06:30:00Z", "Off-peak"),
    ("Winter shoulder start", "2013-01-07 07:00:00Z", "Shoulder"),
    ("Winter before peak", "2013-01-07 15:30:00Z", "Shoulder"),
    ("Winter peak start", "2013-01-07 16:00:00Z", "Peak"),
    ("Winter peak end", "2013-01-07 20:00:00Z", "Shoulder"),
    ("Summer off-peak end", "2013-07-01 05:30:00Z", "Off-peak"),
    ("Summer shoulder start", "2013-07-01 06:00:00Z", "Shoulder"),
    ("Summer peak start", "2013-07-01 15:00:00Z", "Peak"),
    ("Summer peak end", "2013-07-01 19:00:00Z", "Shoulder"),
    ("Weekend afternoon", "2013-07-06 15:00:00Z", "Shoulder"),
    ("Spring clock change", "2013-03-31 01:00:00Z", "Off-peak"),
    ("Autumn first 01:30", "2013-10-27 00:30:00Z", "Off-peak"),
    ("Autumn second 01:30", "2013-10-27 01:30:00Z", "Off-peak"),
], columns=["case", "interval_start_utc", "expected_band"])

local_time = pd.to_datetime(
    time_cases["interval_start_utc"],
    utc=True,
).dt.tz_convert("Europe/London")

time_cases["local_time"] = local_time.astype(str)
time_cases["calculated_band"] = "Shoulder"

time_cases.loc[
    local_time.dt.hour < 7,
    "calculated_band",
] = "Off-peak"

time_cases.loc[
    (local_time.dt.dayofweek < 5)
    & (local_time.dt.hour >= 16)
    & (local_time.dt.hour < 20),
    "calculated_band",
] = "Peak"

time_cases["passed"] = (
    time_cases["calculated_band"] == time_cases["expected_band"]
)

display(time_cases)

assert time_cases["passed"].all(), "Time-band checks failed."

print(f"Time-band boundary scenarios passed: {len(time_cases)}")
print("Scope: classification logic on predefined examples.")
print("Full interval-to-band export reconciliation is still pending.")

,case,interval_start_utc,expected_band,local_time,calculated_band,passed
0,Winter off-peak end,2013-01-07 06:30:00Z,Off-peak,2013-01-07 06:30:00+00:00,Off-peak,True
1,Winter shoulder start,2013-01-07 07:00:00Z,Shoulder,2013-01-07 07:00:00+00:00,Shoulder,True
2,Winter before peak,2013-01-07 15:30:00Z,Shoulder,2013-01-07 15:30:00+00:00,Shoulder,True
3,Winter peak start,2013-01-07 16:00:00Z,Peak,2013-01-07 16:00:00+00:00,Peak,True
4,Winter peak end,2013-01-07 20:00:00Z,Shoulder,2013-01-07 20:00:00+00:00,Shoulder,True
5,Summer off-peak end,2013-07-01 05:30:00Z,Off-peak,2013-07-01 06:30:00+01:00,Off-peak,True
6,Summer shoulder start,2013-07-01 06:00:00Z,Shoulder,2013-07-01 07:00:00+01:00,Shoulder,True
7,Summer peak start,2013-07-01 15:00:00Z,Peak,2013-07-01 16:00:00+01:00,Peak,True
8,Summer peak end,2013-07-01 19:00:00Z,Shoulder,2013-07-01 20:00:00+01:00,Shoulder,True
9,Weekend afternoon,2013-07-06 15:00:00Z,Shoulder,2013-07-06 16:00:00+01:00,Shoulder,True


Time-band boundary scenarios passed: 13
Scope: classification logic on predefined examples.
Full interval-to-band export reconciliation is still pending.


In [4]:
import duckdb
import numpy as np

database_file = processed_folder / "electricity_analysis.duckdb"

with duckdb.connect(
    str(database_file),
    read_only=True,
) as check_connection:
    sql_band_totals = check_connection.execute("""
        WITH localised AS (
            SELECT
                household_id,
                interval_start_utc,
                energy_kwh,
                timezone(
                    'Europe/London',
                    timezone('UTC', interval_start_utc)
                ) AS local_start
            FROM analysis_consumption_2013
            WHERE interval_start_utc >= TIMESTAMP '2013-01-01'
              AND interval_start_utc < TIMESTAMP '2014-01-01'
        ),
        classified AS (
            SELECT
                household_id,
                energy_kwh,
                CASE
                    WHEN interval_start_utc < TIMESTAMP '2013-07-01'
                    THEN 'Design'
                    ELSE 'Validation'
                END AS period,
                CASE
                    WHEN EXTRACT(HOUR FROM local_start) < 7
                    THEN 'Off-peak'
                    WHEN EXTRACT(ISODOW FROM local_start) BETWEEN 1 AND 5
                     AND EXTRACT(HOUR FROM local_start) >= 16
                     AND EXTRACT(HOUR FROM local_start) < 20
                    THEN 'Peak'
                    ELSE 'Shoulder'
                END AS candidate_band
            FROM localised
        )
        SELECT
            period,
            household_id,
            candidate_band,
            COUNT(*) AS intervals,
            SUM(energy_kwh) AS energy_kwh
        FROM classified
        GROUP BY period, household_id, candidate_band
    """).df()

band_check_rows = []

for period, filename, expected_rows in [
    ("Design", "customer_design_band_kwh_2013.csv", 248 * 181 * 48),
    ("Validation", "customer_validation_band_kwh_2013.csv", 248 * 184 * 48),
]:
    period_totals = sql_band_totals.loc[
        sql_band_totals["period"] == period
    ]

    recomputed = period_totals.pivot(
        index="household_id",
        columns="candidate_band",
        values="energy_kwh",
    ).reindex(columns=bands)

    exported = pd.read_csv(
        processed_folder / filename,
        index_col="household_id",
    )

    assert exported.index.is_unique
    assert set(recomputed.index) == set(exported.index)
    assert int(period_totals["intervals"].sum()) == expected_rows

    exported = exported.reindex(
        index=recomputed.index,
        columns=bands,
    )

    assert np.isfinite(recomputed.to_numpy()).all()
    assert np.isfinite(exported.to_numpy()).all()

    differences = (recomputed - exported).abs()

    band_check_rows.append({
        "period": period,
        "households": len(recomputed),
        "intervals": int(period_totals["intervals"].sum()),
        "max_absolute_difference_kwh": differences.to_numpy().max(),
        "band_totals_outside_tolerance": int(
            (differences.to_numpy() > 0.000001).sum()
        ),
    })

band_export_checks = pd.DataFrame(band_check_rows)
display(band_export_checks)

assert (
    band_export_checks["band_totals_outside_tolerance"] == 0
).all(), "Exported band totals require investigation."

print("SQL-to-CSV time-band reconciliation: PASS")
print("Both design and validation periods checked.")
print("Scope: prepared database records to exported band totals.")

,period,households,intervals,max_absolute_difference_kwh,band_totals_outside_tolerance
0,Design,248,2154624,3.183231e-12,0
1,Validation,248,2190336,1.818989e-12,0


SQL-to-CSV time-band reconciliation: PASS
Both design and validation periods checked.
Scope: prepared database records to exported band totals.


In [5]:
validation_folder = project_folder / "reports" / "validation"
validation_folder.mkdir(parents=True, exist_ok=True)

billing_checks.to_csv(
    validation_folder / "billing_reconciliation.csv",
    index=False,
)

time_cases.to_csv(
    validation_folder / "time_band_boundary_checks.csv",
    index=False,
)

band_export_checks.to_csv(
    validation_folder / "sql_band_export_reconciliation.csv",
    index=False,
)

validation_record = {
    "billing_reconciliation": {
        "status": "PASS",
        "households": int(len(recalculated)),
        "fields_checked": int(len(billing_checks)),
        "tolerance_gbp": 0.000001,
        "method": "Independent household-level Decimal calculations"
    },
    "time_band_boundaries": {
        "status": "PASS",
        "scenarios": int(len(time_cases)),
        "coverage": "Weekdays, weekends, seasonal offsets and clock changes"
    },
    "band_export_reconciliation": {
        "status": "PASS",
        "intervals": int(band_export_checks["intervals"].sum()),
        "tolerance_kwh": 0.000001,
        "method": "DuckDB classification compared with exported CSV totals"
    },
    "limitations": [
        "Checks start from the prepared database and saved configuration.",
        "Raw source ingestion was not independently re-audited.",
        "Arithmetic agreement does not validate assumed costs or customer uptake.",
        "Fresh-environment end-to-end reproduction has not yet been verified."
    ]
}

record_file = validation_folder / "calculation_validation_summary.json"

record_file.write_text(
    json.dumps(validation_record, indent=2) + "\n",
    encoding="utf-8",
)

print(f"Validation records saved: {validation_folder}")
print("Completed checks documented with their scope and limitations.")

Validation records saved: D:\electricity-tariff-optimisation\reports\validation
Completed checks documented with their scope and limitations.


In [6]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Notebook files:")
for notebook_path in sorted(
    (project_folder / "notebooks").glob("*.ipynb")
):
    print(f"- {notebook_path.name}")

print(f"\nPython: {sys.version.split()[0]}")

print("\nPackage versions:")
for package in [
    "pandas",
    "numpy",
    "duckdb",
    "scipy",
    "matplotlib",
    "openpyxl",
]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

print("\nEnvironment inventory only; no analysis was rerun.")

Notebook files:
- 01_electricity_data_inspection.ipynb
- 02_customer_sample_preparation.ipynb
- 03_customer_insights.ipynb
- 04_tariff_optimisation.ipynb
- 05_results_visualisation.ipynb

Python: 3.9.15

Package versions:
pandas: 2.2.1
numpy: 1.26.4
duckdb: 1.4.5
scipy: 1.13.0
matplotlib: 3.8.4
openpyxl: 3.1.5

Environment inventory only; no analysis was rerun.


In [7]:
from importlib.metadata import version

packages = [
    "pandas",
    "numpy",
    "duckdb",
    "scipy",
    "matplotlib",
    "openpyxl",
]

requirements_text = "\n".join(
    f"{package}=={version(package)}"
    for package in packages
) + "\n"

requirements_file = project_folder / "requirements.txt"

requirements_file.write_text(
    requirements_text,
    encoding="utf-8",
)

print(requirements_text)
print(f"Requirements saved: {requirements_file}")
print("These are the installed analysis-package versions.")
print("Fresh-environment installation has not yet been verified.")

pandas==2.2.1
numpy==1.26.4
duckdb==1.4.5
scipy==1.13.0
matplotlib==3.8.4
openpyxl==3.1.5

Requirements saved: D:\electricity-tariff-optimisation\requirements.txt
These are the installed analysis-package versions.
Fresh-environment installation has not yet been verified.


In [12]:
import re

readme_file = project_folder / "README.md"

assert readme_file.is_file(), "README.md was not found."

readme_text = readme_file.read_text(encoding="utf-8-sig")

link_targets = re.findall(
    r"!?\[[^\]]*\]\(([^)]+)\)",
    readme_text,
)

missing_targets = []

for target in sorted(set(link_targets)):
    if target.startswith(("https://", "http://", "#")):
        continue

    relative_path = target.split("#", 1)[0]
    target_path = project_folder / relative_path

    if target_path.exists():
        print(f"PASS: {relative_path}")
    else:
        print(f"MISSING: {relative_path}")
        missing_targets.append(relative_path)

requirements_file = project_folder / "requirements.txt"

print(
    "requirements.txt:",
    "PASS" if requirements_file.is_file() else "MISSING",
)

print(f"\nMissing local link targets: {len(missing_targets)}")
print("Local file check only; GitHub upload has not been checked.")

PASS: notebooks/01_electricity_data_inspection.ipynb
PASS: notebooks/02_customer_sample_preparation.ipynb
PASS: notebooks/03_customer_insights.ipynb
PASS: notebooks/04_tariff_optimisation.ipynb
PASS: notebooks/05_results_visualisation.ipynb
MISSING: notebooks/06_calculation_validation.ipynb
PASS: reports/figures/validation_customer_outcomes.png
PASS: reports/figures/validation_savings_by_customer_profile.png
PASS: reports/figures/validation_variable_cost_sensitivity.png
PASS: reports/validation/
requirements.txt: PASS

Missing local link targets: 1
Local file check only; GitHub upload has not been checked.
